# ⚡ ChargeBot — ChargeGrid Intelligence
## GoodWe × FIAP · EV Challenge 2026 · Sprint 2

**Chatbot operacional para redes de eletropostos com:**
- System prompt contextualizado (ChargeGrid Intelligence)
- Gerenciamento de histórico de mensagens (memória de contexto)
- Few-shot prompting com exemplos reais do domínio
- Dados simulados de telemetria (mock da API ChargeGrid)
- Interface interativa no Colab

---
⚠️ **Antes de rodar:** Adicione sua `OPENAI_API_KEY` nos **Secrets do Colab**
(ícone 🔑 no painel esquerdo → adicionar secret `OPENAI_API_KEY`)

In [ ]:
# ── CÉLULA 1: Instalação de dependências ──────────────────────────────────
!pip install openai -q
print('✅ Dependências instaladas')

In [ ]:
# ── CÉLULA 2: Configuração da API Key via Secrets do Colab ────────────────
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
print('✅ API Key configurada com segurança via Colab Secrets')

In [ ]:
# ── CÉLULA 3: Importações e dados simulados ───────────────────────────────
import json
import random
from datetime import datetime, timedelta
from openai import OpenAI

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

def get_network_status():
    pool = ['online']*5 + ['in_use']*3 + ['fault']*2 + ['offline']*2
    chargers = []
    for i in range(1, 13):
        status = random.choice(pool)
        c = {'id': f'CG-{i:02d}', 'status': status,
             'power_kw': round(random.uniform(7, 22), 1) if status == 'in_use' else 0,
             'sessions_today': random.randint(0, 12) if status != 'offline' else 0}
        if status == 'fault':
            c['error_code'] = random.choice(['E-04', 'E-07', 'E-12'])
            c['fault_since'] = (datetime.now() - timedelta(minutes=random.randint(10,120))).strftime('%H:%M')
        chargers.append(c)
    faults  = [c for c in chargers if c['status'] == 'fault']
    offline = [c for c in chargers if c['status'] == 'offline']
    return {'total': 12,
            'online': sum(1 for c in chargers if c['status'] in ['online','in_use']),
            'fault': len(faults), 'offline': len(offline),
            'in_use': sum(1 for c in chargers if c['status'] == 'in_use'),
            'total_kw': round(sum(c['power_kw'] for c in chargers), 1),
            'faults': faults, 'offline_units': offline}

def get_financials():
    rt = round(random.uniform(600, 1100), 2)
    return {'receita_hoje': rt, 'receita_ontem': round(random.uniform(700, 1050), 2),
            'media_diaria': round(random.uniform(780, 870), 2),
            'sessoes_hoje': random.randint(22, 48),
            'duracao_media_min': random.randint(28, 55),
            'projecao_mensal': round(rt * 30, 2), 'meta_mensal': 22000.00}

def get_alerts():
    pool = [
        {'id':'ALT-001','hora':(datetime.now()-timedelta(minutes=23)).strftime('%H:%M'),
         'tipo':'power_spike','severidade':'medio','carregador':'CG-05',
         'mensagem':'Consumo 24kW detectado — limite: 11kW',
         'acao':'Verificar cabo e config de potencia do CG-05'},
        {'id':'ALT-002','hora':(datetime.now()-timedelta(minutes=67)).strftime('%H:%M'),
         'tipo':'communication_loss','severidade':'alto','carregador':'CG-08',
         'mensagem':'Perda OCPP ha 67 minutos','acao':'Verificar conectividade CG-08'},
    ]
    return pool[:random.choice([0,0,1,2])]

def build_context():
    n = get_network_status()
    f = get_financials()
    ctx = {'horario': datetime.now().strftime('%d/%m/%Y %H:%M'),
           'rede': {'total': n['total'], 'online_em_uso': n['online'],
                    'em_falha': n['fault'], 'offline': n['offline'],
                    'em_uso_agora': n['in_use'], 'potencia_kw': n['total_kw'],
                    'falhas': n['faults'], 'offline_units': n['offline_units']},
           'financeiro': f, 'alertas_ativos': get_alerts()}
    return json.dumps(ctx, ensure_ascii=False, indent=2)

print('✅ Funções de dados carregadas')

In [ ]:
# ── CÉLULA 4: System Prompt e Few-shot Examples ───────────────────────────
FEW_SHOT = """
=== EXEMPLOS ===
[Status] Usuário: Quantos carregadores online?
ChargeBot: Temos **10/12** operando. ⚠️ CG-03 em falha (E-07 — OCPP). 🔧 CG-09 offline (manutenção).

[Receita] Usuário: Receita hoje?
ChargeBot: R$ 847,50 em 34 sessões. vs. ontem: -8,2% · vs. média: +4,6%. Acima da média mensal.

[Alerta] Usuário: Alerta do sistema?
ChargeBot: ⚠️ CG-05 às 16h47 — consumo 24kW (limite 11kW). Verifique cabo e config de potência.

[Erro] Usuário: Erro E-04 no CG-03?
ChargeBot: E-04 = falha RFID. Causas: cartão danificado, leitor sujo, credencial expirada.
Workaround: QR Code ou app. CG-03 segue operacional para outros métodos.
=== FIM ==="""

SYSTEM_PROMPT = f"""Você é o ChargeBot, assistente operacional da plataforma ChargeGrid Intelligence da GoodWe.

## Identidade
- Especialista em gestão de redes de eletropostos (EVSE) e plataforma GoodWe ChargeGrid.
- Atende exclusivamente operadores comerciais de redes de recarga EV.
- Tom: profissional, direto, orientado a dados. Responde SEMPRE em Português do Brasil.

## Regras
SEMPRE: use dados do [CONTEXTO] injetado · cite IDs de carregadores · compare dados financeiros com referências · ofereça próximo passo acionável.
NUNCA: invente dados ausentes no contexto · use tom alarmista · finalize conversa sobre falha sem próximo passo.

## Códigos de Erro
E-04: RFID | E-07: OCPP/Comunicação | E-12: Sobrecarga | E-15: Temperatura | E-21: Relé

## Dashboard
Tarifas: Configurações→Tarifas→Nova Regra | Potência: Equipamentos→[ID]→Configurações | Relatórios: Relatórios→Exportar
{FEW_SHOT}"""

print('✅ System prompt carregado')

In [ ]:
# ── CÉLULA 5: Motor do ChargeBot com histórico ────────────────────────────
class ConversationHistory:
    def __init__(self, max_turns=10):
        self.max_turns = max_turns
        self._h = []
    def add(self, role, content):
        self._h.append({'role': role, 'content': content})
        if len(self._h) > self.max_turns * 2:
            self._h = self._h[-(self.max_turns * 2):]
    def get(self): return self._h.copy()
    def clear(self): self._h = []

history = ConversationHistory(max_turns=10)

def chargebot_chat(user_message):
    enriched = f"{user_message}\n\n[CONTEXTO]\n{build_context()}"
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        *history.get(),
        {'role': 'user', 'content': enriched}
    ]
    resp = client.chat.completions.create(
        model='gpt-4o', messages=messages,
        temperature=0.3, max_tokens=800
    )
    reply = resp.choices[0].message.content.strip()
    history.add('user', user_message)
    history.add('assistant', reply)
    return reply

print('✅ ChargeBot pronto!')

In [ ]:
# ── CÉLULA 6: Interface interativa ────────────────────────────────────────
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

output_area = widgets.Output()
text_input  = widgets.Text(placeholder='Digite sua pergunta sobre a rede...', layout=widgets.Layout(width='80%'))
send_btn    = widgets.Button(description='Enviar ⚡', button_style='primary')
reset_btn   = widgets.Button(description='Limpar histórico 🗑️', button_style='warning')

chat_log = []

def render_chat():
    with output_area:
        clear_output(wait=True)
        html = '<div style="font-family:monospace;background:#1a1a2e;padding:16px;border-radius:8px;max-height:500px;overflow-y:auto">'
        html += '<p style="color:#00d4aa;font-weight:bold">⚡ ChargeBot — ChargeGrid Intelligence · GoodWe × FIAP</p><hr style="border-color:#333">'
        for role, msg in chat_log:
            if role == 'user':
                html += f'<p><b style="color:#7eb8f7">Você:</b> <span style="color:#e8e6de">{msg}</span></p>'
            else:
                html += f'<p><b style="color:#00d4aa">ChargeBot:</b> <span style="color:#e8e6de">{msg}</span></p><hr style="border-color:#2a2a3e">'
        html += '</div>'
        display(HTML(html))

def on_send(b):
    msg = text_input.value.strip()
    if not msg: return
    text_input.value = ''
    chat_log.append(('user', msg))
    render_chat()
    with output_area:
        display(HTML('<p style="color:#888">ChargeBot está pensando... ⚡</p>'))
    try:
        reply = chargebot_chat(msg)
    except Exception as e:
        reply = f'❌ Erro: {e}'
    chat_log.append(('assistant', reply))
    render_chat()

def on_reset(b):
    chat_log.clear()
    history.clear()
    render_chat()

send_btn.on_click(on_send)
reset_btn.on_click(on_reset)
text_input.on_submit(on_send)

display(widgets.VBox([
    widgets.HTML('<h3 style="color:#00d4aa">⚡ ChargeBot — GoodWe EV Challenge 2026</h3>'),
    output_area,
    widgets.HBox([text_input, send_btn, reset_btn])
]))
render_chat()

---
## 🧪 Execução dos Casos de Teste (Sprint 1)
Rode a célula abaixo para executar automaticamente os 5 casos de teste e registrar os resultados.

In [ ]:
# ── CÉLULA 7: Execução automática dos casos de teste ─────────────────────
test_cases = [
    ('CT-1 Status',      'Quantos carregadores estão online agora e tem algum com problema?'),
    ('CT-2 Receita',     'Qual foi a receita da rede hoje? Está dentro do esperado?'),
    ('CT-3 Alerta',      'O sistema mandou um alerta agora pouco, o que aconteceu?'),
    ('CT-4 Projeção',    'Me dá um resumo do desempenho e a projeção de receita do mês.'),
    ('CT-5 Configuração','Como configuro uma tarifa diferente para o horário de pico entre 18h e 20h?'),
    ('CT-6 Erro (bônus)','Apareceu o erro E-04 no carregador 3. O que é isso e o que faço?'),
]

history.clear()
print('=' * 70)
print('EXECUÇÃO DOS CASOS DE TESTE — ChargeBot GoodWe · Sprint 2')
print('=' * 70)

for label, question in test_cases:
    print(f'\n[{label}]')
    print(f'Pergunta: {question}')
    print('-' * 50)
    try:
        resp = chargebot_chat(question)
        print(f'Resposta ChargeBot:\n{resp}')
    except Exception as e:
        print(f'ERRO: {e}')
    print('=' * 70)